<a href="https://colab.research.google.com/github/MariaMuu/Thesis/blob/main/Final%20thesis%20with%2Bwikidata%20api%20entity%20search_new%20instructions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install openai

In [10]:
import json
import requests
import os
from openai import OpenAI
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])


# ─── Step 0: Extract entities + relations + intent via OpenAI ─────────────────

def extract_components(question: str) -> dict:
    prompt = f"""
You are an NLP assistant for Wikidata SPARQL query generation.
Given a natural language question, extract the following as JSON:

- entities: list of named entities (people, places, things)
- relations: list of relations/properties being asked about
- intent: one of SELECT, ASK, COUNT
- filters: any constraints (dates, numbers, etc.)
- answer_type: what kind of value is expected (e.g. place, date, person, number)

Question: "{question}"

Respond ONLY with a valid JSON object, no explanation.
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_response_content = response.choices[0].message.content
    # Remove markdown code block delimiters if present
    if raw_response_content.startswith('```json') and raw_response_content.endswith('```'):
        raw_response_content = raw_response_content[len('```json'):-len('```')].strip()
    try:
        return json.loads(raw_response_content)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from OpenAI. Raw response: '{raw_response_content}'")
        raise e

In [18]:
def improve_question_with_llm(question: str) -> str:
    prompt = f"""
    You are an expert at improving natural language questions for search engines.
    Given a question, correct any spelling or grammatical errors, and rephrase it to be clearer and more concise for a factual search, if necessary.
    If the question is already clear and correct, return it as is.

    Example 1:
    Question: "who is the current president of Alban?"
    Improved Question: "who is the current president of Albania?"

    Example 2:
    Question: "tell me about capital of france"
    Improved Question: "What is the capital of France?"

    Example 3:
    Question: "what is the population of the city with the eiffel tower"
    Improved Question: "What is the population of Paris?"

    Question: "{question}"
    Improved Question:
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

In [19]:
# ─── Step 1: Link entities to Wikidata Q/P numbers via Falcon 2.0 ─────────────

def link_to_wikidata(question: str) -> dict:
    url = "https://labs.tib.eu/falcon/falcon2/api?mode=short"
    headers = {"Content-Type": "application/json"}
    payload = {"text": question}
    response = requests.post(url, headers=headers, json=payload)
    data = response.json()

    entities = [
        {"label": e.get("label"), "uri": e.get("uri")}
        for e in data.get("entities_wikidata", [])
    ]
    relations = [
        {"label": r.get("label"), "uri": r.get("uri")}
        for r in data.get("relations_wikidata", [])
    ]
    return {"entities": entities, "relations": relations}

In [20]:
# ─── Step 2: Linking entities to Wikidata via Wikidata API ─────────────────

# import requests

# def link_entity_to_qid(entity_name: str, language: str = "en") -> list:
#     """
#     Use the Wikidata search API to find QID candidates for an entity name.
#     Returns a ranked list of candidates with QID, label, and description.
#     """
#     url = "https://www.wikidata.org/w/api.php"
#     params = {
#         "action": "wbsearchentities",
#         "search": entity_name,
#         "language": language,
#         "format": "json",
#         "limit": 5
#     }
#     # Add a User-Agent header
#     headers = {
#         "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
#     }
#     response = requests.get(url, params=params, headers=headers)

#     # Always print status code and response content for debugging
#     print(f"Wikidata API Response Status Code: {response.status_code}")
#     print(f"Wikidata API Response Content: {response.text}")

#     # Raise an HTTPError for bad responses (4xx or 5xx)
#     response.raise_for_status()

#     try:
#         if not response.text.strip(): # Check if the response content is empty or only whitespace
#             print("Wikidata API returned an empty or whitespace-only response.")
#             return [] # Return empty list if no content to parse
#         data = response.json()
#     except requests.exceptions.JSONDecodeError as e:
#         print(f"JSONDecodeError from Wikidata API after status check. Raw response text: '{response.text}'")
#         raise e

#     candidates = []
#     for result in data.get("search", []):
#         candidates.append({
#             "qid": result["id"],
#             "label": result.get("label", ""),
#             "description": result.get("description", ""),
#         })
#     return candidates
# print(link_entity_to_qid("simulation"))

Wikidata API Response Status Code: 200
Wikidata API Response Content: {"searchinfo":{"search":"simulation"},"search":[{"id":"Q1610017","title":"Q1610017","pageid":1544805,"concepturi":"http://www.wikidata.org/entity/Q1610017","repository":"wikidata","url":"//www.wikidata.org/wiki/Q1610017","display":{"label":{"value":"simulation video game","language":"en"},"description":{"value":"video game genre","language":"en"}},"label":"simulation video game","description":"video game genre","match":{"type":"label","language":"en","text":"simulation video game"}},{"id":"Q45045","title":"Q45045","pageid":47216,"concepturi":"http://www.wikidata.org/entity/Q45045","repository":"wikidata","url":"//www.wikidata.org/wiki/Q45045","display":{"label":{"value":"simulation","language":"en"},"description":{"value":"Imitation of the operation of a real-world process or system over time","language":"en"}},"label":"simulation","description":"Imitation of the operation of a real-world process or system over time"

In [21]:
# ─── Step 3: Generate SPARQL query via OpenAI ─────────────────────────────────

def generate_sparql(question: str, components: dict, linked: dict) -> str:
    prompt = f"""
You are a SPARQL expert for Wikidata.
Generate a valid Wikidata SPARQL query for the following question.

Question: "{question}"

Extracted components:
{json.dumps(components, indent=2)}

Wikidata linked entities and relations:
{json.dumps(linked, indent=2)}

Rules:
- Use the Wikidata SPARQL endpoint format (wd:, wdt:, p:, ps:, pq:)
- Use SERVICE wikibase:label for labels
- Use LIMIT 10 unless a specific count is requested
- NEVER use property paths with * or + (e.g. wdt:P171*) — they time out
- NEVER use deeply nested subqueries
- Keep queries flat: maximum 3-4 triple patterns per query
- For multi-hop questions, generate only the FIRST hop query
  and note which variable needs a second query
- For time-sensitive properties (current officeholder, current population),
  use the qualifier pattern:
    p:PXX ?stmt . ?stmt ps:PXX ?value .
    FILTER NOT EXISTS {{ ?stmt pq:P582 ?end }}
- Return ONLY the SPARQL query, no explanation

SPARQL:
"""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_sparql_query = response.choices[0].message.content.strip()
    # Remove markdown code block delimiters if present
    if raw_sparql_query.startswith('```sparql') and raw_sparql_query.endswith('```'):
        raw_sparql_query = raw_sparql_query[len('```sparql'):-len('```')].strip()
    return raw_sparql_query

In [22]:
# ─── Step 4: Run the SPARQL query against Wikidata ────────────────────────────

def run_sparql(query: str) -> list:
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
    }
    response = requests.get(url, params={"query": query}, headers=headers)
    print(f"Raw SPARQL response status: {response.status_code}")
    print(f"Raw SPARQL response text: {response.text}")
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return data["results"]["bindings"]


In [23]:
# ─── Step 4: Run the SPARQL query against Wikidata ────────────────────────────

def run_sparql(query: str) -> list:
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "Colab Wikidata Query Agent/1.0 (Python Requests)"
    }
    response = requests.get(url, params={"query": query}, headers=headers)
    print(f"Raw SPARQL response status: {response.status_code}")
    print(f"Raw SPARQL response text: {response.text}")
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return data["results"]["bindings"]


# ─── Main pipeline ────────────────────────────────────────────────────────────

def answer_question(question: str):
    print(f"\nQuestion: {question}\n")

    print("Step 1: Extracting components...")
    components = extract_components(question)
    print(json.dumps(components, indent=2))

    print("\nStep 2: Linking to Wikidata...")
    linked = link_to_wikidata(question)
    print(json.dumps(linked, indent=2))

    print("\nStep 3: Generating SPARQL...")
    sparql = generate_sparql(question, components, linked)
    print(sparql)

    print("\nStep 4: Querying Wikidata...")
    results = run_sparql(sparql)
    print(f"Got {len(results)} result(s):")
    for r in results:
        for key, val in r.items():
            print(f"  {key}: {val['value']}")

    return results

### Answering the question using the Wikidata pipeline

In [34]:
# Define the question once here
user_question = "Name 9 countries in Europe"
user_question_llm = improve_question_with_llm(user_question)
print(f"Original Question: {user_question}")
print(f"Improved Question (LLM): {user_question_llm}")

Original Question: Name 9 countries in Europe
Improved Question (LLM): "Name nine countries in Europe."


In [35]:
print(f"\nRunning Wikidata pipeline for question: {user_question_llm}\n")
wikidata_raw_results = answer_question(user_question_llm)


Running Wikidata pipeline for question: "Name nine countries in Europe."


Question: "Name nine countries in Europe."

Step 1: Extracting components...
{
  "entities": [
    "Europe"
  ],
  "relations": [
    "countries"
  ],
  "intent": "SELECT",
  "filters": {
    "limit": 9
  },
  "answer_type": "place"
}

Step 2: Linking to Wikidata...
{
  "entities": [
    {
      "label": null,
      "uri": null
    }
  ],
  "relations": []
}

Step 3: Generating SPARQL...
SELECT ?country ?countryLabel WHERE {
  ?country wdt:P31 wd:Q6256;  # instance of country
           wdt:P30 wd:Q46.    # located in Europe
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
LIMIT 9

Step 4: Querying Wikidata...
Raw SPARQL response status: 200
Raw SPARQL response text: {
  "head" : {
    "vars" : [ "country", "countryLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "country" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q29999"
      },

### Instructions to the LLM re its answers

In [38]:
def ask_llm_directly(question: str) -> str:
    prompt = f"""
    Answer the following question without repeating user's question in your answer.

    Question: "{question}"

    Answer:
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

### User's question

In [39]:
if __name__ == "__main__":
    question_for_llm = user_question_llm
    print(f"\nQuestion for LLM: {question_for_llm}\n")
    llm_direct_answer = ask_llm_directly(question_for_llm)
    print(f"LLM's Answer: {llm_direct_answer}")


Question for LLM: "Name nine countries in Europe."

LLM's Answer: France, Germany, Italy, Spain, United Kingdom, Netherlands, Belgium, Switzerland, and Sweden.


### LLM-as-a-Judge Evaluation

In [42]:
def llm_judge(question: str, wikidata_result: list, llm_answer: str) -> dict:
    # Format Wikidata results for the LLM prompt
    formatted_wikidata_answers = []
    if wikidata_result:
        for binding in wikidata_result:
            found_label = False
            # First, try to find any key ending with 'Label'
            for key, value_dict in binding.items():
                if key.endswith('Label') and 'value' in value_dict:
                    formatted_wikidata_answers.append(value_dict['value'])
                    found_label = True
                    break

            # If no label was found for this binding, fallback to the first 'value' found in any key
            if not found_label:
                for key, value_dict in binding.items():
                    if 'value' in value_dict:
                        formatted_wikidata_answers.append(value_dict['value'])
                        break

    # If no answers were extracted, set a default message
    formatted_wikidata_answer_str = ", ".join(formatted_wikidata_answers) if formatted_wikidata_answers else "No answer found from Wikidata."

    prompt = f"""
    You are an expert evaluator for factual questions. Your task is to act as an LLM-as-a-judge.
    Given an original question and two answers (one obtained from Wikidata and one from a direct LLM call), perform the following:

    1. Judge each answer independently first. Do NOT let one answer influence scoring of the other.
    2. Evaluate the factual accuracy of each answer (Wikidata and direct LLM) on a scale of 0-100%.
    2. Provide a brief explanation for each accuracy score, noting any specifics about the answer (e.g., if it's too brief, too verbose, or incorrect).
    3. If both answers are highly accurate (e.g., > 80%), provide a combined, concise answer to the original question. This combined answer should be more than one word, but not overly verbose, and synthesize the information.
    4. If one answer is significantly more accurate, favor that one for the combined answer.
    5. Do not assume agreement implies correctness. Both answers may be wrong.
    6. If neither is accurate, state that a combined answer is not possible.
    7. Penalize hallucinations and unsupported specifics.

    --------------------------------------------------
    SCORING RUBRIC (0-100)
    --------------------------------------------------

    Score each answer using:

    1. Factual Correctness (0-70)
    - Is it true?
    - Any fabricated claims?
    - Any contradictions?

    2. Completeness (0-20)
    - Does it fully answer the question?
    - Missing key parts?

    3. Relevance / Clarity (0-10)
    - Directly answers question?
    - Clear and concise?

    Total = sum (0-100)

    --------------------------------------------------
    FINAL ANSWER RULES
    --------------------------------------------------

    If one answer scores at least 15 points higher than the other, prefer that answer.

    If both answers score >= 80 and are compatible, synthesize a concise better answer.

    If one answer is correct but incomplete, and the other adds correct useful details, merge them carefully.

    If both answers are weak (<60), set combined_answer = null.

    If uncertainty remains, say so briefly.

    Never copy false claims into the combined answer.

    --------------------------------------------------
    OUTPUT FORMAT
    --------------------------------------------------

    Return ONLY valid JSON.

    {{
      "user_question": "...",

      "wikidata_answer_provided": "...",
      "wikidata_accuracy_percent": 0,
      "wikidata_breakdown": {{
        "factual_correctness": 0,
        "completeness": 0,
        "relevance_clarity": 0
      }},
      "wikidata_explanation": "...",

      "llm_direct_answer_provided": "...",
      "llm_accuracy_percent": 0,
      "llm_breakdown": {{
        "factual_correctness": 0,
        "completeness": 0,
        "relevance_clarity": 0
      }},
      "llm_explanation": "...",

      "winner": "wikidata | llm | merged | none",

      "confidence": 0.0,

      "combined_answer": "...",

      "needs_human_review": false
    }}

    Original Question: "{question}"
    Wikidata Answer: "{formatted_wikidata_answer_str}"
    Direct LLM Answer: "{llm_answer}"

    Respond ONLY with a valid JSON object, no explanation.
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw_response_content = response.choices[0].message.content
    # Remove markdown code block delimiters if present
    if raw_response_content.startswith('```json') and raw_response_content.endswith('```'):
        raw_response_content = raw_response_content[len('```json'):-len('```')].strip()
    try:
        return json.loads(raw_response_content)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from LLM Judge. Raw response: '{raw_response_content}'")
        raise e

print(f"\n\n--- LLM-as-a-Judge Evaluation ---\n")
judge_results = llm_judge(user_question_llm, wikidata_raw_results, llm_direct_answer)
print(json.dumps(judge_results, indent=2))



--- LLM-as-a-Judge Evaluation ---

{
  "user_question": "Name nine countries in Europe.",
  "wikidata_answer_provided": "Kingdom of the Netherlands, Kingdom of Navarre, Kosovo, Northern Cyprus, Norway, Ireland, Hungary, Spain, Belgium",
  "wikidata_accuracy_percent": 50,
  "wikidata_breakdown": {
    "factual_correctness": 30,
    "completeness": 10,
    "relevance_clarity": 10
  },
  "wikidata_explanation": "The answer includes some entities that are not recognized as current countries in Europe, such as the Kingdom of Navarre and Northern Cyprus. Kosovo's status is disputed. The rest are valid European countries.",
  "llm_direct_answer_provided": "France, Germany, Italy, Spain, United Kingdom, Netherlands, Belgium, Switzerland, and Sweden.",
  "llm_accuracy_percent": 100,
  "llm_breakdown": {
    "factual_correctness": 70,
    "completeness": 20,
    "relevance_clarity": 10
  },
  "llm_explanation": "All listed entities are recognized as countries in Europe, and the answer is compl